<a href="https://colab.research.google.com/github/Mmbsaksd/transformers/blob/main/Annotated_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Annotated Transformer — Simple Notes

## 1. Background

Older models such as Extended Neural GPU, ByteNet, and ConvS2S also tried to reduce step-by-step computation. They used CNNs to process many positions at the same time.

However, when two words were far apart, these models needed more computation to connect them. This made it harder to learn relationships between distant words.

The Transformer uses **self-attention**, which allows words to directly connect with other words, even when they are far apart.

**Self-attention = a word can look at other words in the same sequence to understand their relationship.**

The Transformer was the first sequence-to-sequence model that relied completely on self-attention instead of RNNs or CNNs.

---

# 2. Model Architecture

Most sequence-to-sequence models have two main parts:

```text
Input
  ↓
Encoder
  ↓
Information
  ↓
Decoder
  ↓
Output
```

### Encoder

The Encoder takes the input tokens and converts them into numerical representations.

```text
Input tokens → Encoder → Encoder representations
```

For example:

```text
"I love you"
     ↓
  Encoder
     ↓
z₁, z₂, z₃
```

Each `z` is a vector containing information about the corresponding input position and its context.

### Decoder

The Decoder uses the Encoder's information to generate the output.

It generates the output **one token at a time**.

```text
Encoder information
       ↓
    Decoder
       ↓
token 1 → token 2 → token 3 → ...
```

The Decoder is **autoregressive**, which means:

> When generating the next token, it uses the tokens that it has already generated.

---

# 3. `EncoderDecoder` Class

The `EncoderDecoder` class connects the main parts of the Transformer:

```text
Source
  ↓
Source Embedding
  ↓
Encoder
  ↓
Memory
  ↓
Decoder
  ↓
Generator
  ↓
Output
```

### Main components

```python
self.encoder
```

Processes the source/input.

```python
self.decoder
```

Generates the target/output using the encoder information.

```python
self.src_embed
```

Converts source token IDs into vectors.

```python
self.tgt_embed
```

Converts target token IDs into vectors.

```python
self.generator
```

Converts the decoder's output into scores for the vocabulary.

---

# 4. `forward()`

```python
return self.decode(
    self.encode(src, src_mask),
    src_mask,
    tgt,
    tgt_mask
)
```

The main flow is:

```text
src
 ↓
encode()
 ↓
Encoder output
 ↓
decode()
 ↓
Decoder output
```

So:

> **First encode the source, then use that information to decode the target.**

---

# 5. `encode()`

```python
def encode(self, src, src_mask):
    return self.encoder(self.src_embed(src), src_mask)
```

The source tokens first go through the embedding:

```text
src
 ↓
src_embed
 ↓
vectors
 ↓
Encoder
```

`src_mask` tells the Encoder which source positions should be ignored, mainly **padding positions**.

The Encoder is not using `src_mask` to hide future words.

---

# 6. `decode()`

```python
def decode(self, memory, src_mask, tgt, tgt_mask):
    return self.decoder(
        self.tgt_embed(tgt),
        memory,
        src_mask,
        tgt_mask
    )
```

The target tokens are converted into vectors:

```text
tgt
 ↓
tgt_embed
 ↓
target vectors
```

The Decoder then uses:

* target vectors
* Encoder output (`memory`)
* `src_mask`
* `tgt_mask`

The target mask prevents the Decoder from seeing future tokens and also handles target padding.

---

# 7. Generator

```python
self.proj = nn.Linear(d_model, vocab)
```

The Decoder produces a vector of size `d_model`.

The Generator converts that vector into scores for every token in the vocabulary.

```text
Decoder output
      ↓
Linear layer
      ↓
Vocabulary scores
      ↓
log_softmax
      ↓
Score/probability for each token
```

If the vocabulary contains 30,000 tokens, the Generator produces a score for each of those 30,000 possible tokens.

```python
return log_softmax(self.proj(x), dim=-1)
```

Here:

```python
self.proj(x)
```

produces the vocabulary scores.

Then:

```python
log_softmax(..., dim=-1)
```

converts those scores into **log-probabilities**.

`dim=-1` means:

> Apply softmax across the last dimension, which is the vocabulary dimension.

---

# 8. Overall Transformer Architecture

The Transformer uses **self-attention and feed-forward networks** instead of RNNs or CNNs.

```text
                 INPUT
                   ↓
             Embedding
                   ↓
              Encoder
        ┌──────────┴──────────┐
        │                     │
   Self-Attention       Feed-Forward
        │                     │
        └────── Encoder ──────┘
                   ↓
                Memory
                   ↓
              Decoder
        ┌──────────┴──────────┐
        │          │          │
   Self-Attention  │   Encoder-Decoder
                   │      Attention
                   │          │
                   └── Feed-Forward
                   ↓
               Generator
                   ↓
                Output
```

---

# 9. Encoder Stack

The Encoder is made of **6 identical layers** in the original Transformer:

```text
Input
 ↓
Encoder Layer 1
 ↓
Encoder Layer 2
 ↓
Encoder Layer 3
 ↓
Encoder Layer 4
 ↓
Encoder Layer 5
 ↓
Encoder Layer 6
 ↓
Encoder output
```

Each layer has the same structure.

---

# 10. `clones()`

```python
def clones(module, N):
    return nn.ModuleList(
        [copy.deepcopy(module) for _ in range(N)]
    )
```

This creates `N` copies of a layer.

For example:

```python
clones(layer, 6)
```

creates:

```text
Layer 1
Layer 2
Layer 3
Layer 4
Layer 5
Layer 6
```

### PyTorch part

```python
copy.deepcopy(module)
```

creates a separate copy of the module.

```python
nn.ModuleList(...)
```

tells PyTorch:

> These are multiple neural-network modules that belong to my model.

Each layer has the **same structure**, but each copy has its **own learnable parameters**.

---

# 11. Encoder Layer — Two Sub-Layers

Each Encoder layer has **two sub-layers**:

### Sub-layer 1

**Multi-Head Self-Attention**

It allows each token to look at other tokens in the input.

```text
Input
 ↓
Self-Attention
 ↓
Context-aware information
```

### Sub-layer 2

**Position-wise Feed-Forward Network**

It processes each position's representation further.

```text
Attention output
 ↓
Feed-Forward Network
 ↓
Processed representation
```

So:

```text
Encoder Layer

Input
 ↓
Multi-Head Self-Attention
 ↓
Feed-Forward Network
 ↓
Output
```

---

# 12. Residual Connection

Each sub-layer uses a residual connection.

The basic flow is:

```text
Input
  ↓
Sub-layer
  ↓
Dropout
  ↓
+ original Input
  ↓
LayerNorm
  ↓
Output
```

The formula is:

```text
LayerNorm(Input + Dropout(Sublayer(Input)))
```

### Residual connection

The original input is added back to the sub-layer output.

> **It helps preserve the original information and makes deep networks easier to train.**

---

# 13. Layer Normalization

LayerNorm normalizes the values in each token's feature vector.

```text
Input
 ↓
Calculate mean
 ↓
Calculate standard deviation
 ↓
Normalize
 ↓
Learnable scale
 ↓
Learnable shift
 ↓
Output
```

The code:

```python
self.a_2 = nn.Parameter(torch.ones(features))
```

creates a **learnable scale**.

```python
self.b_2 = nn.Parameter(torch.zeros(features))
```

creates a **learnable shift**.

```python
mean = x.mean(-1, keepdim=True)
```

calculates the mean across the **last dimension**, which is the feature dimension.

```python
std = x.std(-1, keepdim=True)
```

calculates the standard deviation across the feature dimension.

```python
(x - mean) / (std + self.eps)
```

normalizes the values.

Finally:

```python
self.a_2 * normalized + self.b_2
```

allows the model to learn the best scale and shift.

---

# 14. Dropout

Dropout is applied to the sub-layer output **before** the residual addition.

```text
Sub-layer
   ↓
Dropout
   ↓
+ original input
   ↓
LayerNorm
```

During training, dropout randomly removes some values.

> **Purpose: reduce overfitting and help the model generalize better.**

---

# 15. `d_model = 512`

The original Transformer uses:

```text
d_model = 512
```

This means each token's main representation has **512 features**.

For example:

```text
"I"
 ↓
[0.2, -0.4, 0.7, ... 512 values ...]
```

The important reason for using the same size everywhere is the residual connection.

We need to add:

```text
Input + Sub-layer output
```

So both should have the same feature size:

```text
512 + 512
```

Therefore:

> **All major sub-layers and embedding layers produce 512-dimensional representations.**

---

# 16. Complete Encoder Layer

Putting everything together:

```text
                  Input
                    │
                    ├─────────────────────┐
                    ↓                     │
          Multi-Head Self-Attention       │
                    ↓                     │
                 Dropout                 │
                    ↓                     │
                    └──────── + ─────────┘
                              ↓
                         LayerNorm
                              ↓
                    ┌─────────┴─────────┐
                    │                   │
                    ↓                   │
             Feed-Forward              │
                    ↓                   │
                 Dropout                │
                    ↓                   │
                    └──────── + ────────┘
                              ↓
                         LayerNorm
                              ↓
                            Output
```

The output then goes to the **next Encoder layer**.

---

# 17. Most Important Things to Remember

```text
Encoder
    ↓
6 identical layers
    ↓
Each layer has:
    ↓
1. Multi-Head Self-Attention
2. Feed-Forward Network
    ↓
Each sub-layer has:
    ↓
Sub-layer
 ↓
Dropout
 ↓
Residual connection
 ↓
LayerNorm
```

And the overall Transformer is:

```text
Input
 ↓
Embedding
 ↓
Encoder × 6
 ↓
Encoder output / Memory
 ↓
Decoder × 6
 ↓
Linear + Softmax
 ↓
Output
```

### In one sentence:

> **The Transformer takes the input, creates embeddings, processes them through 6 Encoder layers using self-attention and feed-forward networks, and then the Decoder uses this information to generate the output one token at a time.**


# **Prelims**

In [3]:
# # Uncomment for colab
# #
!pip install -q  GPUtil
!python -m spacy download de_core_news_sm
!python -m spacy download en_core_web_sm

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 41.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 36.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
import os
from os.path import exists
import torch
import torch.nn as nn
from torch.nn.functional import log_softmax, pad
import math
import copy
import time
from torch.optim.lr_scheduler import LambdaLR
import pandas as pd
import altair as alt
from torch.utils.data import DataLoader, Dataset
import spacy
import GPUtil
import warnings
from torch.utils.data.distributed import DistributedSampler
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from collections import Counter
from datasets import load_dataset

warnings.filterwarnings("ignore")
RUN_EXAMPLES = True

In [5]:
def is_interactive_notebook():
    return __name__ == "__main__"


def show_example(fn, args=[]):
    if __name__ == "__main__" and RUN_EXAMPLES:
        return fn(*args)


def execute_example(fn, args=[]):
    if __name__ == "__main__" and RUN_EXAMPLES:
        fn(*args)


class DummyOptimizer(torch.optim.Optimizer):
    def __init__(self):
        self.param_groups = [{"lr": 0}]
        None

    def step(self):
        None

    def zero_grad(self, set_to_none=False):
        None


class DummyScheduler:
    def step(self):
        None

### Background

Earlier models such as Extended Neural GPU, ByteNet, and ConvS2S also tried to reduce step-by-step computation. They used convolutional neural networks (CNNs) to process many positions in the input at the same time.

However, these models had a problem when two words were far apart in a sentence. The farther apart the words were, the more computation was needed to connect them. ConvS2S needed more computation as the distance increased, while ByteNet increased more slowly but still needed additional computation.

The Transformer solves this problem using **self-attention**. Self-attention allows any word to directly look at other words in the sequence, even when they are far apart. This makes it easier for the model to learn relationships between distant words.

Self-attention was already used in some NLP tasks before the Transformer. However, the Transformer was the first model to use self-attention as the main mechanism for processing both the input and output, without using RNNs or CNNs.

In simple terms:

**Older models:** Farther words → more computation to connect them.

**Transformer:** Farther words → can still directly connect through self-attention.


## **Part 1: Model Architecture**

## Model Architecture

Most sequence-to-sequence models use two main parts: an **Encoder** and a **Decoder**.

The **Encoder** takes the input sequence and converts it into useful numerical representations. For example, the input words are converted into vectors, and the Encoder uses the relationships between the words to create better representations.

The **Decoder** uses these representations to generate the output sequence. It generates the output **one token at a time**. When generating the next token, it can use the tokens it has already generated.

The Transformer follows this Encoder-Decoder structure, but instead of using RNNs or CNNs to process the sequence, it mainly uses **attention mechanisms**.

The Encoder is made up of several identical layers. Each layer contains two main parts:

1. **Multi-Head Self-Attention** – allows each word to look at other words in the input and understand their relationships.
2. **Feed-Forward Network** – processes the information from the attention layer further.

The Decoder is also made up of several identical layers. Each layer contains three main parts:

1. **Masked Multi-Head Self-Attention** – allows the decoder to look at previously generated words, but prevents it from looking at future words.
2. **Encoder-Decoder Attention** – allows the decoder to look at the information produced by the Encoder.
3. **Feed-Forward Network** – processes the information further.

The Transformer also uses **residual connections and normalization** around these parts to help the network train effectively.

In simple terms:

**Input → Encoder → Information → Decoder → Output**

The main idea is that the **Encoder understands the input**, and the **Decoder uses that information to generate the output one token at a time**.


In [6]:
class EncoderDecoder(nn.Module):
    """
    A standard Encoder-Decoder architecture. Base for this and many
    other models.
    """

    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super(EncoderDecoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.generator = generator

    def forward(self, src, tgt, src_mask, tgt_mask):
        "Take in and process masked src and target sequences."
        return self.decode(self.encode(src, src_mask), src_mask, tgt, tgt_mask)

    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)

In [23]:
class Generator(nn.Module):
    "Define standard linear + softmax generation step."

    def __init__(self, d_model, vocab):
        super(Generator, self).__init__()
        self.proj = nn.Linear(d_model, vocab)

    def forward(self, x):
        return log_softmax(self.proj(x), dim=-1)

## **Encoder and Decoder Stacks**
### Encoder
The encoder is composed of a stack of  N=6  identical layers

In [7]:
def clones(module, N):
    "Produce N identical layers."
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [8]:
class Encoder(nn.Module):
    "Core encoder is a stack of N layers"

    def __init__(self, layer, N):
        super(Encoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, mask):
        "Pass the input (and mask) through each layer in turn."
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

### Residual Connections and Layer Normalization

Each sub-layer in the Transformer uses a **residual connection** followed by **Layer Normalization**.

A residual connection adds the original input to the output of the sub-layer:

```text
Input → Sub-layer → Output
  └──────────────────┘
          +
```

In simple terms:

> **Residual connection = keep the original information and add it to the new information.**

After adding them, **Layer Normalization** is applied to keep the values in a stable range and help the model train better.

The Encoder has two sub-layers:

1. Multi-Head Self-Attention
2. Feed-Forward Network

Both use the same pattern:

```text
Input
  ↓
Sub-layer
  ↓
Add original input
  ↓
Layer Normalization
  ↓
Next layer
```

So, the main idea is:

> **Residual connection helps preserve the original information, while Layer Normalization helps the model train more smoothly.**


In [9]:
class LayerNorm(nn.Module):
    "Construct a layernorm module (See citation for details)."

    def __init__(self, features, eps=1e-6):
        super(LayerNorm, self).__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

### Residual Connection, Dropout, and `d_model`

Each Transformer sub-layer follows this basic flow:

```text
Input
  ↓
Sub-layer
  ↓
Dropout
  ↓
Add original Input
  ↓
Layer Normalization
  ↓
Output
```

The calculation is:

```text
Output = LayerNorm(Input + Dropout(Sublayer(Input)))
```

* **Residual connection:** The original input is added back to the sub-layer output. This helps preserve the original information.
* **Dropout:** Some values from the sub-layer output are randomly dropped during training. This helps reduce overfitting.
* **Layer Normalization:** The result is normalized to make training more stable.
* **`d_model = 512`:** Every major representation in the Transformer has 512 features. This is important because the original input and the sub-layer output need compatible dimensions for the residual addition.

For example:

```text
Input:          512 features
Sub-layer:      512 features
       ↓
    Add them
       ↓
Output:         512 features
```

So the main idea is:

> **Process the input → apply dropout → add the original input → normalize the result.**


In [10]:
class SublayerConnection(nn.Module):
    """
    A residual connection followed by a layer norm.
    Note for code simplicity the norm is first as opposed to last.
    """

    def __init__(self, size, dropout):
        super(SublayerConnection, self).__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        "Apply residual connection to any sublayer with the same size."
        return x + self.dropout(sublayer(self.norm(x)))

### Encoder Layer

Each Encoder layer has **two sub-layers**:

1. **Multi-Head Self-Attention** – allows each token to look at other tokens in the input sequence and understand their relationships.

2. **Position-wise Feed-Forward Network** – further processes the representation of each token independently.

In simple terms:

> **Self-Attention understands relationships between tokens, and the Feed-Forward Network processes each token's information further.**


In [11]:
class EncoderLayer(nn.Module):
    "Encoder is made up of self-attn and feed forward (defined below)"

    def __init__(self, size, self_attn, feed_forward, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 2)
        self.size = size

    def forward(self, x, mask):
        "Follow Figure 1 (left) for connections."
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask))
        return self.sublayer[1](x, self.feed_forward)

### Decoder

The decoder is also composed of a stack of $N=6$ identical layers.

In [12]:
class Decoder(nn.Module):
    "Generic N layer decoder with masking."

    def __init__(self, layer, N):
        super(Decoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)

### Decoder Layer

Each **Decoder layer has three sub-layers**:

1. **Masked Multi-Head Self-Attention** – allows the Decoder to look at previously generated target tokens, but not future tokens.

2. **Encoder-Decoder Multi-Head Attention** – allows the Decoder to look at the **Encoder's output** and use the relevant information from the input sequence.

3. **Position-wise Feed-Forward Network** – further processes the information for each token.

Each sub-layer uses the same:

```text
Sub-layer
   ↓
Dropout
   ↓
Residual Connection
   ↓
Layer Normalization
```

So the Decoder layer is:

```text
Target
  ↓
Masked Self-Attention
  ↓
Encoder-Decoder Attention
  ↓
Feed-Forward Network
  ↓
Output
```

> **In simple terms: The Decoder first looks at the previous target tokens, then looks at the Encoder's output to find useful input information, and finally processes that information further.**


In [13]:
class DecoderLayer(nn.Module):
    "Decoder is made of self-attn, src-attn, and feed forward (defined below)"

    def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
        super(DecoderLayer, self).__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 3)

    def forward(self, x, memory, src_mask, tgt_mask):
        "Follow Figure 1 (right) for connections."
        m = memory
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask))
        return self.sublayer[2](x, self.feed_forward)


### Decoder Self-Attention Masking

In the Decoder's **self-attention**, we prevent a position from looking at **future positions**.

For example, when predicting the next token:

```text
I → love → you
```

When predicting **"you"**, the Decoder can use:

```text
I ✓
love ✓
you ✗  ← future/current answer is hidden
```

This is done using a **mask**.

The target sequence is also shifted by one position, so the Decoder receives the **previous tokens** as input and predicts the next token.

```text
Decoder input:   <START>   I       love
                   ↓        ↓        ↓
Predicts:           I      love      you
```

Therefore:

> **The Decoder can only use tokens that are already known when predicting the next token. It cannot look at future tokens.**

In simple terms:

**Masking + shifting the target by one position = prevents the Decoder from cheating by seeing the correct future answer.**


In [14]:
def subsequent_mask(size):
    "Mask out subsequent positions."
    attn_shape = (1, size, size)
    subsequent_mask = torch.triu(torch.ones(attn_shape), diagonal=1).type(
        torch.bool
    )
    return subsequent_mask == 0

In [15]:
def example_mask():
    LS_data = pd.concat(
        [
            pd.DataFrame(
                {
                    "Subsequent Mask": subsequent_mask(20)[0][x, y].flatten(),
                    "Window": y,
                    "Masking": x,
                }
            )
            for y in range(20)
            for x in range(20)
        ]
    )

    return (
        alt.Chart(LS_data)
        .mark_rect()
        .properties(height=250, width=250)
        .encode(
            alt.X("Window:O"),
            alt.Y("Masking:O"),
            alt.Color("Subsequent Mask:Q", scale=alt.Scale(scheme="viridis")),
        )
        .interactive()
    )


show_example(example_mask)

alt.Chart(...)

## Attention

Attention helps the model decide **which information is important**.

It takes:

* **Query (Q)** – what we are looking for
* **Keys (K)** – information we can compare with the query
* **Values (V)** – the actual information we want to use

The model compares the **Query with every Key** to find how relevant each one is.

It then gives higher weight to more relevant Values and combines them to produce the final output.

```text
Query + Keys
     ↓
Calculate how relevant they are
     ↓
Attention weights
     ↓
Weighted combination of Values
     ↓
Output
```

### Scaled Dot-Product Attention

The Transformer uses a specific type of attention called **Scaled Dot-Product Attention**.

The basic steps are:

```text
1. Compare Query with every Key using dot product
2. Divide the scores by √dₖ
3. Apply Softmax to convert scores into attention weights
4. Use these weights to combine the Values
5. Get the final attention output
```

In simple terms:

> **The Query asks "What am I looking for?", the Keys help find which information is relevant, and the Values provide the actual information.**

The formula is:

```text
Attention(Q, K, V) = softmax(QKᵀ / √dₖ)V
```

Where:

* `Q` = Queries
* `K` = Keys
* `V` = Values
* `dₖ` = dimension (number of features) of the Keys
* `√dₖ` = scaling factor used to keep the attention scores stable.


## Attention Using Matrices

In practice, we calculate attention for **all tokens at the same time** instead of calculating one token at a time.

The Query vectors are stored together in a matrix **Q**. Similarly, all Key vectors are stored in **K**, and all Value vectors are stored in **V**.

For example, if we have **7 tokens** and each vector has **4 features**:

```text
Q = 7 × 4
K = 7 × 4
V = 7 × 4
```

Each row represents one token.

### Step 1: Compare every Query with every Key

We calculate:

```text
Q × Kᵀ
```

```text
(7 × 4) × (4 × 7)
        ↓
      7 × 7
```

The resulting **7 × 7 matrix** contains the relationship/compatibility score between every Query and every Key.

```text
Rows    → Queries
Columns → Keys
```

So, for example:

```text
Row 1 → Query of token 1 compared with all 7 Keys
Row 2 → Query of token 2 compared with all 7 Keys
...
Row 7 → Query of token 7 compared with all 7 Keys
```

A **higher score means stronger relationship** between that Query and Key.

### Step 2: Scale the scores

We divide the scores by:

```text
√dₖ
```

This keeps the values at a reasonable scale.

### Step 3: Apply Softmax

Softmax converts the scores into **attention weights**.

```text
Raw scores
    ↓
Softmax
    ↓
Attention weights
```

The weights tell us:

> **How much attention should each Query give to each Key?**

The formula for the complete process is:

```text
Attention(Q, K, V) = softmax(QKᵀ / √dₖ)V
```

### Simple flow

```text
Q, K, V
  ↓
Q × Kᵀ
  ↓
Relationship scores
  ↓
÷ √dₖ
  ↓
Softmax
  ↓
Attention weights
  ↓
× V
  ↓
Attention Output
```

> **In simple words: We compare every Query with every Key, convert the comparison scores into attention weights, and use those weights to combine the Values and produce the final attention output.**


In [16]:
def attention(query, key, value, mask=None, dropout=None):
    "Compute 'Scaled Dot Product Attention'"
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    p_attn = scores.softmax(dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn

### Why do we scale Dot-Product Attention?

There are two common attention methods: **Additive Attention** and **Dot-Product Attention**.

The Transformer uses **Scaled Dot-Product Attention** because dot-product attention is faster and more memory-efficient on GPUs, as it can be implemented using optimized matrix multiplication.

However, when the Key/Query dimension (`dₖ`) becomes large, the dot-product scores can become very large. Large scores can make Softmax produce very extreme values, such as `[0, 0, 1]`, which results in very small gradients and makes learning difficult.

To solve this, we divide the dot-product scores by `√dₖ` before applying Softmax:

Attention(Q, K, V) = softmax(QKᵀ / √dₖ)V

The scaling keeps the attention scores at a reasonable range and prevents Softmax from becoming too extreme.

**Simple idea:**

Q × Kᵀ → large scores → divide by √dₖ → reasonable scores → Softmax → attention weights.

## Multi-Head Attention

Multi-Head Attention allows the model to look at the same input from **different representation spaces at the same time**.

Instead of using one large attention operation, the model divides the representation into multiple smaller attention heads. Each head can learn different relationships between tokens.

For example, one head may learn word relationships, while another may learn grammatical or long-distance relationships. The model learns these patterns automatically during training.

In the original Transformer:

- `d_model = 512`
- Number of heads `h = 8`
- Each head uses `d_k = d_v = 512 / 8 = 64`

So instead of one attention operation using 512 dimensions, we have 8 smaller attention operations using 64 dimensions each.

```text
Q, K, V
   ↓
Split into 8 heads
   ↓
Head 1 → Attention
Head 2 → Attention
Head 3 → Attention
...
Head 8 → Attention
   ↓
Concatenate all heads
   ↓
7 × 512
   ↓
Linear projection (Wᵒ)
   ↓
Multi-Head Attention Output

In [17]:
class MultiHeadedAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.1):
        "Take in model size and number of heads."
        super(MultiHeadedAttention, self).__init__()
        assert d_model % h == 0
        # We assume d_v always equals d_k
        self.d_k = d_model // h
        self.h = h
        self.linears = clones(nn.Linear(d_model, d_model), 4)
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        "Implements Figure 2"
        if mask is not None:
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)

        query, key, value = [
            lin(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2)
            for lin, x in zip(self.linears, (query, key, value))
        ]

        x, self.attn = attention(
            query, key, value, mask=mask, dropout=self.dropout
        )

        x = (
            x.transpose(1, 2)
            .contiguous()
            .view(nbatches, -1, self.h * self.d_k)
        )
        del query
        del key
        del value
        return self.linears[-1](x)

### Applications of Attention in the Transformer

The Transformer uses Multi-Head Attention in three different ways:

1. **Encoder Self-Attention**
   - Q, K, and V come from the encoder.
   - Each token can attend to all tokens in the input.

2. **Decoder Self-Attention**
   - Q, K, and V come from the decoder.
   - The decoder can attend to previous and current tokens.
   - Future tokens are blocked using a mask.

3. **Encoder-Decoder Attention**
   - Q comes from the decoder.
   - K and V come from the encoder.
   - This allows the decoder to look at the encoder output and focus on the relevant parts of the input.

```text
Encoder Self-Attention
Q = Encoder
K = Encoder
V = Encoder

Decoder Self-Attention
Q = Decoder
K = Decoder
V = Decoder
+ Future Mask

Encoder-Decoder Attention
Q = Decoder
K = Encoder
V = Encoder

## Position-wise Feed-Forward Networks

After the attention layer, each token goes through a Feed-Forward Network (FFN).

The FFN processes each token separately.

Token → Linear → ReLU → Linear → Output

The Transformer uses:

d_model = 512
d_ff = 2048

So the dimensions change like this:

512 → 2048 → 512

The same FFN is used for every token in the same layer.

Different layers have different FFN parameters.

Attention → Tokens interact with each other

FFN → Each token is processed separately

In [18]:
class PositionwiseFeedForward(nn.Module):
    "Implements FFN equation."

    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionwiseFeedForward, self).__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.w_2(self.dropout(self.w_1(x).relu()))

# Embeddings and Softmax

The Transformer uses learned embeddings to convert token IDs into vectors.

## 1. Token Embedding

Each token is first converted into a token ID.

```text
Sentence
   ↓
Tokenization
   ↓
Token IDs
   ↓
Embedding Layer
   ↓
512-dimensional vectors
```

For the Transformer:

```text
d_model = 512
```

So, every token is represented by a vector containing 512 numbers.

Example:

```text
Token ID: 25
    ↓
Embedding Layer
    ↓
[0.2, 0.7, -0.1, ..., 512 numbers]
```

The embedding values are learned during training.

## 2. Embedding Scaling

The Transformer multiplies the embedding values by √d_model.

```text
Token Embedding
      ↓
Multiply by √d_model
      ↓
Scaled Embedding
```

For the base Transformer:

```text
d_model = 512

Embedding × √512
```

This changes the values but does not change the dimension.

```text
Before scaling  → 512 dimensions
After scaling   → 512 dimensions
```

## 3. Decoder Output → Next Token Probability

After the decoder processes the sequence, it produces a vector for each position.

```text
Decoder
   ↓
512-dimensional output
   ↓
Linear Layer
   ↓
Vocabulary-size scores
   ↓
Softmax
   ↓
Probability for each token
```

For example, if the vocabulary contains 10,000 tokens:

```text
Decoder Output
      ↓
     512
      ↓
Linear Layer
      ↓
   10,000 scores
      ↓
    Softmax
      ↓
10,000 probabilities
```

Example:

```text
cat   → 0.60
dog   → 0.20
car   → 0.10
book  → 0.05
...   → ...
```

The token with the highest probability can be selected as the next token.

## 4. Shared Weight Matrix

The Transformer shares the same weight matrix between:

```text
Encoder Embedding
       ↓
   Same Weights
       ↑
Decoder Embedding

       and

Final Linear Layer
       ↑
   Same Weights
```

In simple terms:

```text
One shared weight matrix
        ↓
Used by:
    • Encoder Embedding
    • Decoder Embedding
    • Final Linear Layer
```

This reduces the number of separate parameters and allows the same learned token representations to be reused.

## Complete Flow

```text
Input Sentence
      ↓
Tokenization
      ↓
Token IDs
      ↓
Embedding Layer
      ↓
512-dimensional Embeddings
      ↓
Multiply by √512
      ↓
Positional Encoding
      ↓
Encoder
      ↓
Decoder
      ↓
512-dimensional Decoder Output
      ↓
Final Linear Layer
      ↓
Vocabulary-size Scores
      ↓
Softmax
      ↓
Probability of Each Possible Next Token
      ↓
Next Token
```

## Simple Summary

```text
Embedding:
Token ID → 512-dimensional vector

Scaling:
Embedding → Embedding × √512

Decoder:
Produces 512-dimensional output

Linear + Softmax:
512-dimensional output
        ↓
Vocabulary scores
        ↓
Next-token probabilities
```

In [19]:
class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super(Embeddings, self).__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model

    def forward(self, x):
        return self.lut(x) * math.sqrt(self.d_model)

## Positional Encoding

The Transformer does not use RNNs or CNNs, so it does not automatically know the order of the tokens.

For example:

I → position 0
love → position 1
cats → position 2

To give the model position information, we add Positional Encoding to the token embeddings.

Token Embedding + Positional Encoding
                    ↓
          Final Representation

Both have the same dimension:

d_model = 512

So:

Embedding          → 512 dimensions
Positional Encoding → 512 dimensions
                      ↓
                     Add
                      ↓
                   512 dimensions

The positional encoding gives each position a different pattern of numbers.

Position 0 → Pattern 0
Position 1 → Pattern 1
Position 2 → Pattern 2
Position 3 → Pattern 3

Therefore, even if the same word appears in different positions, its final representation will be different.

Example:

Same word + Position 0 → Different representation
Same word + Position 1 → Different representation
Same word + Position 2 → Different representation

### Sine and Cosine

The Transformer creates these positional patterns using sine and cosine functions.

Even dimensions → Sine
Odd dimensions  → Cosine

PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))

PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

Where:

pos → position of the token
i   → dimension index

Different dimensions use different frequencies. This creates a unique pattern for each position.

These patterns help the model learn the position of tokens and the relationship between their positions.

### Dropout

After adding the token embedding and positional encoding, dropout is applied.

Embedding
    +
Positional Encoding
    ↓
Combined Representation
    ↓
Dropout
    ↓
Encoder / Decoder

For the base Transformer:

P_drop = 0.1

Dropout randomly turns off some values during training to help prevent overfitting.

### Simple Summary

Sentence
   ↓
Tokens
   ↓
Token Embeddings
   ↓
Add Positional Encoding
   ↓
Dropout
   ↓
Transformer

Embedding          → What is the token?
Positional Encoding → Where is the token?

In [20]:
class PositionalEncoding(nn.Module):
    "Implement the PE function."

    def __init__(self, d_model, dropout, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Compute the positional encodings once in log space.
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, : x.size(1)].requires_grad_(False)
        return self.dropout(x)

In [21]:
def example_positional():
    pe = PositionalEncoding(20, 0)
    y = pe.forward(torch.zeros(1, 100, 20))

    data = pd.concat(
        [
            pd.DataFrame(
                {
                    "embedding": y[0, :, dim],
                    "dimension": dim,
                    "position": list(range(100)),
                }
            )
            for dim in [4, 5, 6, 7]
        ]
    )

    return (
        alt.Chart(data)
        .mark_line()
        .properties(width=800)
        .encode(x="position", y="embedding", color="dimension:N")
        .interactive()
    )


show_example(example_positional)

alt.Chart(...)

# **Full Model**

In [22]:
def make_model(
    src_vocab, tgt_vocab, N=6, d_model=512, d_ff=2048, h=8, dropout=0.1
):
    "Helper: Construct a model from hyperparameters."
    c = copy.deepcopy
    attn = MultiHeadedAttention(h, d_model)
    ff = PositionwiseFeedForward(d_model, d_ff, dropout)
    position = PositionalEncoding(d_model, dropout)
    model = EncoderDecoder(
        Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout), N),
        Decoder(DecoderLayer(d_model, c(attn), c(attn), c(ff), dropout), N),
        nn.Sequential(Embeddings(d_model, src_vocab), c(position)),
        nn.Sequential(Embeddings(d_model, tgt_vocab), c(position)),
        Generator(d_model, tgt_vocab),
    )

    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    return model


In [24]:
def inference_test():
    test_model = make_model(11, 11, 2)
    test_model.eval()
    src = torch.tensor([[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]], dtype=torch.long)
    src_mask = torch.ones(1, 1, 10)

    memory = test_model.encode(src, src_mask)

    ys = torch.zeros(1, 1).type_as(src)

    for i in range(9):
        out = test_model.decode(
            memory, src_mask, ys, subsequent_mask(ys.size(1)).type_as(src.data)
        )
        prob = test_model.generator(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.data[0]
        ys = torch.cat(
            [ys, torch.full((1, 1), next_word, dtype=ys.dtype)], dim=1
        )

    print("Example Untrained Model Prediction:", ys)

def run_tests():
    for _ in range(10):
        inference_test()


show_example(run_tests)


Example Untrained Model Prediction: tensor([[ 0,  1, 10,  9,  1, 10,  9,  1, 10,  9]])
Example Untrained Model Prediction: tensor([[0, 4, 1, 6, 7, 6, 7, 1, 2, 3]])
Example Untrained Model Prediction: tensor([[0, 1, 1, 1, 6, 3, 3, 3, 3, 1]])
Example Untrained Model Prediction: tensor([[0, 6, 1, 6, 1, 6, 1, 6, 1, 6]])
Example Untrained Model Prediction: tensor([[ 0,  6,  0,  2, 10, 10,  1,  3,  6,  0]])
Example Untrained Model Prediction: tensor([[0, 4, 4, 5, 4, 5, 4, 4, 4, 5]])
Example Untrained Model Prediction: tensor([[0, 3, 8, 0, 3, 8, 0, 3, 3, 3]])
Example Untrained Model Prediction: tensor([[0, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
Example Untrained Model Prediction: tensor([[0, 2, 6, 1, 1, 1, 1, 6, 8, 6]])
Example Untrained Model Prediction: tensor([[ 0,  8,  7,  7,  7,  7,  7, 10,  3,  7]])


In [25]:
def inference_test():
    print("--- Starting inference test ---")

    # Create and evaluate the model
    test_model = make_model(11, 11, 2)
    test_model.eval()
    print("Model created and set to evaluation mode.")

    # Input sequence: numbers 1 to 10
    src = torch.tensor([[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]], dtype=torch.long)
    src_mask = torch.ones(1, 1, 10)
    print(f"Input sequence (src): {src}")

    # Encode the input sequence
    memory = test_model.encode(src, src_mask)
    print("Input sequence encoded into memory.")

    # Initialize output sequence with a single zero
    ys = torch.zeros(1, 1).type_as(src)
    print(f"Initial output sequence (ys): {ys}")

    # Generate predictions for the next 9 tokens
    for i in range(9):
        print(f"\n--- Decoding step {i+1} ---")

        # Decode the current output sequence
        out = test_model.decode(
            memory,
            src_mask,
            ys,
            subsequent_mask(ys.size(1)).type_as(src.data)
        )
        print("Decoded output.")

        # Generate probabilities for the next token
        prob = test_model.generator(out[:, -1])
        print("Generated probabilities for the next token.")

        # Select the token with the highest probability
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.data[0]
        print(f"Selected next token: {next_word}")

        # Append the selected token to the output sequence
        ys = torch.cat(
            [ys, torch.full((1, 1), next_word, dtype=ys.dtype)],
            dim=1
        )
        print(f"Updated output sequence (ys): {ys}")

    print("\n--- Final Prediction ---")
    print("Example Untrained Model Prediction:", ys)

def run_tests():
    print("\n--- Running 10 inference tests ---")
    for _ in range(10):
        inference_test()

show_example(run_tests)


--- Running 10 inference tests ---
--- Starting inference test ---
Model created and set to evaluation mode.
Input sequence (src): tensor([[ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10]])
Input sequence encoded into memory.
Initial output sequence (ys): tensor([[0]])

--- Decoding step 1 ---
Decoded output.
Generated probabilities for the next token.
Selected next token: 2
Updated output sequence (ys): tensor([[0, 2]])

--- Decoding step 2 ---
Decoded output.
Generated probabilities for the next token.
Selected next token: 10
Updated output sequence (ys): tensor([[ 0,  2, 10]])

--- Decoding step 3 ---
Decoded output.
Generated probabilities for the next token.
Selected next token: 3
Updated output sequence (ys): tensor([[ 0,  2, 10,  3]])

--- Decoding step 4 ---
Decoded output.
Generated probabilities for the next token.
Selected next token: 10
Updated output sequence (ys): tensor([[ 0,  2, 10,  3, 10]])

--- Decoding step 5 ---
Decoded output.
Generated probabilities for the next token.
